In [ ]:
# Install required packages

!pip install -q google-generativeai
!pip install -q ipywidgets

# Imports

import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configure the API

api_key = "YOUR_API_KEY_HERE"  # <-- Replace with your valid API key
genai.configure(api_key=api_key)

# List available models that support chat

print("Available models with 'generateContent':")
available_models = []
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        available_models.append(model.name)
        print(f"- {model.name}")

# Select a default model from available ones

# Replace with any model from the printed list if needed
default_model_name = available_models[0] if available_models else "models/chat-bison-001"
model = genai.GenerativeModel(default_model_name)
chat = model.start_chat()

# Create interactive chat interface

chat_output = widgets.Output()
input_text = widgets.Text(placeholder="Type your message here...")
send_button = widgets.Button(description="Send")
clear_button = widgets.Button(description="Clear Chat")

model_dropdown = widgets.Dropdown(
    options=available_models,
    value=default_model_name,
    description="Model:"
)

# Functions for model change and chat

def change_model(change):
    global model, chat
    model = genai.GenerativeModel(change['new'])
    chat = model.start_chat()
    with chat_output:
        print(f"Model changed to: {change['new']}")
        print("New chat session started!")

model_dropdown.observe(change_model, names='value')

def on_send_clicked(b):
    user_input = input_text.value.strip()
    if user_input:
        with chat_output:
            print(f"You: {user_input}")
            try:
                response = chat.send_message(user_input)
                print(f"Bot: {response.text}\n")
            except Exception as e:
                print(f"Error: {str(e)}\n")
        input_text.value = ""

def on_clear_clicked(b):
    global chat
    chat = model.start_chat()
    chat_output.clear_output()
    with chat_output:
        print("Chat cleared. New session started!")

send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)

# Display the interface

display(widgets.VBox([
    model_dropdown,
    widgets.HBox([input_text, send_button, clear_button]),
    chat_output
]))

# Initial greeting

with chat_output:
    print("Bot: Hello! I'm Vishal's chatbot. How can I help you today?")
